# Simple RAG (Retrieval-Augmented Generation) with Ollama

A minimal, end-to-end RAG pipeline in one notebook:

1. **Load** a small sample knowledge base (a fictional company's help docs)
2. **Chunk** the documents
3. **Embed** the chunks (TF-IDF vectors, so there is nothing to download)
4. **Retrieve** the top-k chunks for a question with cosine similarity
5. **Generate** an answer with a local LLM, grounded in the retrieved chunks

**Prerequisites**
- [Ollama](https://ollama.com) running with a model pulled: `ollama pull llama3.2`
- If Ollama isn't reachable, the notebook falls back to showing the retrieved context so retrieval can still be explored.

In [5]:
%pip install -q scikit-learn numpy requests


[notice] A new release of pip is available: 24.3.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 1. Sample data
Six short documents for a fictional company, *Acme Cloud*. In a real system these would come from files, a wiki, a database, etc.

In [1]:
DOCUMENTS = [
    {"source": "refund-policy.md", "text": """
Acme Cloud Refund Policy

Customers may request a full refund within 30 days of purchase for any annual plan. Monthly plans are refundable within 7 days of the first charge. Refunds are issued to the original payment method and typically take 5-10 business days to appear. Usage-based charges for compute and storage already consumed are non-refundable. To request a refund, open a ticket in the Billing category and include your invoice number.
"""},
    {"source": "pricing.md", "text": """
Acme Cloud Pricing

The Free plan includes 1 project, 5 GB of storage, and community support. The Pro plan costs $29 per user per month and includes unlimited projects, 500 GB of storage, and email support with a 24-hour response time. The Enterprise plan has custom pricing and adds single sign-on (SSO), audit logs, a 99.99% uptime SLA, and a dedicated account manager. Annual billing gives a 20% discount on Pro.
"""},
    {"source": "security.md", "text": """
Acme Cloud Security

All data is encrypted at rest with AES-256 and in transit with TLS 1.3. Acme Cloud is SOC 2 Type II and ISO 27001 certified. Two-factor authentication (2FA) is available on every plan and can be enforced organization-wide by admins. Enterprise customers can bring their own encryption keys (BYOK). Security incidents are disclosed to affected customers within 72 hours. Vulnerabilities can be reported to the security team through the bug bounty program.
"""},
    {"source": "api-limits.md", "text": """
Acme Cloud API Limits

The REST API allows 100 requests per minute on the Free plan, 1,000 requests per minute on Pro, and 10,000 requests per minute on Enterprise. Exceeding the limit returns HTTP 429 with a Retry-After header. Requests must be authenticated with an API key sent in the Authorization header as a Bearer token. API keys can be rotated at any time from Settings > Developer. The maximum request payload is 10 MB.
"""},
    {"source": "support.md", "text": """
Acme Cloud Support

Support is available through the in-app chat and the help center. Free plan users get community forum support only. Pro users receive email support with a 24-hour response time, Monday to Friday. Enterprise users get 24/7 phone and chat support with a 1-hour response time for critical incidents. The status page at status.acmecloud.example shows current outages and scheduled maintenance.
"""},
    {"source": "data-retention.md", "text": """
Acme Cloud Data Retention

When an account is cancelled, project data is kept for 30 days so it can be restored. After 30 days all data is permanently deleted from primary storage, and from backups within 90 days. Customers can export their data at any time as JSON or CSV from Settings > Data. Deleted projects can be restored from the trash within 14 days. Audit logs on the Enterprise plan are retained for 1 year.
"""},
]
print(f"{len(DOCUMENTS)} documents loaded")

6 documents loaded


## 2. Chunking
Split each document into small overlapping word windows. Smaller chunks give more precise retrieval; overlap avoids cutting an answer in half.

In [3]:
def chunk_text(text, size=60, overlap=15):
    words = text.split()
    step = size - overlap
    return [" ".join(words[i:i + size]) for i in range(0, max(len(words) - overlap, 1), step)]

chunks = []  # each: {"id", "source", "text"}
for doc in DOCUMENTS:
    for piece in chunk_text(doc["text"]):
        chunks.append({"id": len(chunks), "source": doc["source"], "text": piece})

print(f"{len(chunks)} chunks")
print(chunks[0])

12 chunks
{'id': 0, 'source': 'refund-policy.md', 'text': 'Acme Cloud Refund Policy Customers may request a full refund within 30 days of purchase for any annual plan. Monthly plans are refundable within 7 days of the first charge. Refunds are issued to the original payment method and typically take 5-10 business days to appear. Usage-based charges for compute and storage already consumed are non-refundable. To request a refund,'}


## 3. Embed and index
TF-IDF turns each chunk into a sparse vector. It's keyword-based, so it's a simple stand-in for a neural embedding model (see the swap-in note at the end).

In [6]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), sublinear_tf=True)
chunk_matrix = vectorizer.fit_transform([c["text"] for c in chunks])
print("index shape:", chunk_matrix.shape)

index shape: (12, 423)


## 4. Retrieval
Embed the question the same way, then rank chunks by cosine similarity (TF-IDF vectors are L2-normalised, so a dot product is cosine).

In [8]:
def retrieve(question, k=3):
    q_vec = vectorizer.transform([question])
    scores = (chunk_matrix @ q_vec.T).toarray().ravel()
    top = np.argsort(scores)[::-1][:k]
    return [{**chunks[i], "score": float(scores[i])} for i in top if scores[i] > 0]

for r in retrieve("How long do I have to get my money back?"):
    print(f"{r['score']:.3f}  [{r['source']}]  {r['text'][:90]}...")

## 5. Generation
Build a prompt that includes the retrieved chunks and tells the model to answer **only** from them. This grounding is what makes it RAG.

In [10]:
import requests

OLLAMA_URL = "http://localhost:11434"
MODEL = "llama3.2"

PROMPT = """You are a support assistant for Acme Cloud.
Answer the question using ONLY the context below. If the answer is not in the context, say "I don't know based on the available documents."

Context:
{context}

Question: {question}
Answer:"""

def build_prompt(question, hits):
    context = "\n\n".join(f"[{h['source']}] {h['text']}" for h in hits)
    return PROMPT.format(context=context, question=question)

def llm(prompt):
    r = requests.post(f"{OLLAMA_URL}/api/chat", json={
        "model": MODEL,
        "messages": [{"role": "user", "content": prompt}],
        "stream": False,
        "options": {"temperature": 0},
    }, timeout=120)
    r.raise_for_status()
    return r.json()["message"]["content"].strip()

def ask(question, k=3, show_context=True):
    hits = retrieve(question, k)
    if show_context:
        print("Retrieved:")
        for h in hits:
            print(f"  {h['score']:.3f}  {h['source']}")
    if not hits:
        return "I don't know based on the available documents."
    prompt = build_prompt(question, hits)
    try:
        answer = llm(prompt)
    except requests.RequestException as e:
        answer = f"(Ollama unavailable: {e.__class__.__name__}. Top retrieved chunk:)\n{hits[0]['text']}"
    return answer

## 6. Try it

In [11]:
questions = [
    "Can I get a refund on my annual plan?",
    "What is the API rate limit on the Pro plan?",
    "Is my data encrypted?",
    "How long is my data kept after I cancel?",
    "Does Acme Cloud offer a free GPU for training models?",   # not in the docs -> should say "I don't know"
]
for q in questions:
    print("Q:", q)
    print("A:", ask(q))
    print("-" * 70)

Q: Can I get a refund on my annual plan?
Retrieved:
  0.271  refund-policy.md
  0.082  refund-policy.md
  0.076  pricing.md
A: Yes, you can request a full refund within 30 days of purchase for your annual plan.
----------------------------------------------------------------------
Q: What is the API rate limit on the Pro plan?
Retrieved:
  0.219  api-limits.md
  0.135  pricing.md
  0.124  api-limits.md
A: The API rate limit on the Pro plan is 1,000 requests per minute.
----------------------------------------------------------------------
Q: Is my data encrypted?
Retrieved:
  0.200  security.md
  0.125  data-retention.md
  0.072  data-retention.md
A: Yes, all data is encrypted at rest with AES-256 and in transit with TLS 1.3.
----------------------------------------------------------------------
Q: How long is my data kept after I cancel?
Retrieved:
  0.291  data-retention.md
  0.072  data-retention.md
  0.045  security.md
A: After you cancel, your project data is kept for 30 days so i

## Ideas to extend

- **Neural embeddings:** replace TF-IDF with `sentence-transformers` (`all-MiniLM-L6-v2`) or Ollama's `nomic-embed-text` (`POST /api/embed`). Only `chunk_matrix` and `retrieve` change; the rest stays the same.
- **Vector store:** swap the NumPy matrix for Chroma, FAISS or pgvector once you have more than a few thousand chunks.
- **Better chunking:** split on headings/sentences instead of fixed word windows.
- **Evaluate:** write question/expected-source pairs and measure retrieval hit-rate@k before tuning the prompt.
- **Cite sources:** ask the model to name the `[source]` files it used.